# 1. Overview and environment

This notebook rebuilds the **version 7** submission, public Kaggle score **0.724** (user reported, uploaded
18 Nov 2026), from the four competition files in `data/raw/`. Local scores on the team's test sets: 0.7038 (fixed)
and 0.7284 (mirror); these are local only.

| Mission | Version 7 method | Code |
|---|---|---|
| FORECAST | 0.5 x accepted T05 forecast + 0.5 x timing-shift forecast; for cameras B, C, D, G, H, 0.25 of that value is replaced by Chronos-2 | `noran/tracks/forecast/`, `noran/tracks/lib_forecast/` |
| HIDDEN | five-member log-scale blend with CatBoost and a hedge for B, plus TabICL v2 for B (0.33) and D (0.20) | `noran/tracks/hidden_v4/` |
| HIDDEN-FUTURE | the camera's own curve from the same weekday last week, aligned to today's prayer times and scaled to today's level | `noran/tracks/hf_final/` |
| MISSING | straight line across the gap, corrected by the partner camera and the camera's own level | `noran/tracks/missing/` |

The editable settings are in `configs/version7.json`; each mission section shows its part and may change it. The
frozen inputs, assets, package versions, pretrained models and the accepted fingerprint are in
`models/version7/manifest.json`; the notebook stops on any mismatch. Each step runs its script in a separate
Python process and logs to `outputs/version7/logs/`.

**Kernel:** the version 7 environment (`requirements_version7.txt`, see `docs/version7.md`). All steps use the
kernel's interpreter unless the environment variable `MADINAH_PYTHON` names another one.
**Runtime:** about 13 minutes on a laptop CPU (measured 19 Nov 2026), plus a one-time download of about 570 MB of model weights.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time

import numpy as np
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "t05_forecast_config.json").is_file() and (p / "noran").is_dir()), None)
assert ROOT is not None, "Open the notebook from inside the repository."
PY = Path(os.environ.get("MADINAH_PYTHON", sys.executable))
MANIFEST = json.loads((ROOT / "models/version7/manifest.json").read_text(encoding="utf-8"))
OUT = ROOT / "outputs/version7"
LOGS = OUT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
print("repository:", ROOT)
print("interpreter:", PY)

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# Package versions of the verified build. A different version can change the values, so the notebook stops.
probe = "import importlib.metadata as m, json, sys; v = {}\n" \
        "for n in sys.argv[1:]:\n    try: v[n] = m.version(n)\n    except Exception: v[n] = None\nprint(json.dumps(v))"
found = json.loads(subprocess.run([str(PY), "-c", probe, *MANIFEST["versions"]],
                                  capture_output=True, text=True, check=True).stdout)
versions = pd.DataFrame({"manifest": MANIFEST["versions"], "found": found})
display(versions)
wrong = versions.index[versions.manifest != versions.found].tolist()
assert not wrong, f"Package versions differ from models/version7/manifest.json: {wrong}. Install requirements_version7.txt."

In [ ]:
# Pretrained models: downloaded once at the pinned revision, then checked file by file against the manifest.
fetch = '''
import json, sys
from huggingface_hub import hf_hub_download
repo, rev, names = sys.argv[1], sys.argv[2], sys.argv[3:]
paths = {}
for name in names:
    try:
        paths[name] = hf_hub_download(repo, name, revision=rev, local_files_only=True)
    except Exception:
        paths[name] = hf_hub_download(repo, name, revision=rev)
print(json.dumps(paths))
'''
for key, spec in MANIFEST["pretrained_models"].items():
    result = subprocess.run([str(PY), "-c", fetch, spec["repo"], spec["revision"], *spec["files"]],
                            capture_output=True, text=True)
    assert result.returncode == 0, result.stderr[-2000:]
    for name, path in json.loads(result.stdout).items():
        assert sha256(path) == spec["files"][name], f"{key}/{name}: SHA-256 differs from the manifest"
    print(f"{key:10s} {spec['repo']} @ {spec['revision'][:8]}  OK  ({spec['licence']}; {spec['used_for']})")

# 2. Load and check the data

Place `train.csv`, `test.csv`, `recordings.csv` and `sample_submission.csv` in `data/raw/`. The cell checks the
four files and the frozen assets (prayer timetable and the twelve T05 covariance files in
`models/current_baseline/`) against `models/version7/manifest.json`, then counts the targets of each mission.

In [ ]:
DATA = ROOT / "data/raw"
for name, digest in MANIFEST["inputs"].items():
    assert sha256(DATA / name) == digest, f"data/raw/{name}: SHA-256 differs from the manifest"
for name, digest in MANIFEST["assets"].items():
    assert sha256(ROOT / name) == digest, f"{name}: SHA-256 differs from the manifest"
print(f"{len(MANIFEST['inputs'])} input files and {len(MANIFEST['assets'])} frozen assets match the manifest")

test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
targets = test.dropna(subset=["id"])
display(targets.groupby(["day", "kind"]).size().unstack(fill_value=0))

# 3. Shared processing and settings

`CONFIG` is read from `configs/version7.json`. Each mission section below copies its part into an editable
dictionary; `save_settings()` writes the settings actually used to `outputs/version7/task_config.json`.

Every mission starts from the accepted 0.806 baseline file, rebuilt here by `noran/run_baseline_local.py`
(the code path of `notebooks/current_baseline.ipynb`). A mission module returns a value for each target it
changes; every other target keeps the baseline value. `run_step` runs one script with `PYTHONPATH` set to the
repository root and the Hugging Face cache offline, and stops the notebook if the script fails.
Runtime of this section: about 1 minute.

In [ ]:
CONFIG = json.loads((ROOT / "configs/version7.json").read_text(encoding="utf-8"))
TIMES = {}

def run_step(name, *args, env=None):
    environment = dict(os.environ, PYTHONPATH=str(ROOT), HF_HUB_OFFLINE="1", PYTHONUTF8="1", **(env or {}))
    log = LOGS / f"{name}.log"
    started = time.perf_counter()
    with open(log, "w", encoding="utf-8") as handle:
        result = subprocess.run([str(PY), "-X", "utf8", *map(str, args)], cwd=ROOT, env=environment,
                                stdout=handle, stderr=subprocess.STDOUT)
    TIMES[name] = round(time.perf_counter() - started, 1)
    if result.returncode != 0:
        raise RuntimeError(f"{name} failed with exit code {result.returncode}; see {log}")
    print(f"{name}: {TIMES[name]:.0f} s")

def save_settings():
    """Write the settings used by this run, as edited in sections 4 to 7."""
    used = dict(CONFIG, forecast=FORECAST_CONFIG, hidden=HIDDEN_CONFIG,
                **{"hidden-future": HIDDEN_FUTURE_CONFIG, "missing": MISSING_CONFIG})
    path = OUT / "task_config.json"
    path.write_text(json.dumps(used, indent=2), encoding="utf-8")
    return path

def mission_summary(label_or_path, kind):
    path = ROOT / (label_or_path if str(label_or_path).endswith(".csv")
                   else f"outputs/noran/tracks/{label_or_path}/test_predictions.csv")
    values = pd.read_csv(path).set_index("id").people
    rows = targets[targets.kind.eq(kind)].set_index("id")[["day", "unit"]]
    frame = rows.assign(new=values.reindex(rows.index), baseline=baseline.reindex(rows.index))
    assert frame.new.notna().all() and np.isfinite(frame.new).all() and (frame.new >= 0).all()
    frame["changed"] = (frame.new - frame.baseline).abs() > 1e-9
    print(f"{path.relative_to(ROOT)}: {len(frame)} targets, {int(frame.changed.sum())} changed against the baseline")
    return frame.groupby(["day", "unit"]).agg(targets=("new", "size"), changed=("changed", "sum"),
                                               mean_baseline=("baseline", "mean"), mean_new=("new", "mean")).round(2)

run_step("03_accepted_baseline", "noran/run_baseline_local.py", "outputs/noran/baseline_local")
baseline = pd.read_csv(ROOT / "outputs/noran/baseline_local/submission_baseline_local.csv").set_index("id").people
print("accepted baseline rows:", len(baseline))

# 4. FORECAST

Predicts each visible camera 1 to 15 minutes after `now`, from every camera's readings up to `now` only.
Version 7 combines three forecasts:

- the accepted T05 forecast (from the baseline file);
- the timing-shift forecast: cameras A, E and F over the last 45 minutes measure how many minutes today runs
  early or late against last week, then T05 runs with that shift and the frozen covariances in
  `models/current_baseline/t05/` (nothing is fitted); cameras I to M keep the accepted rule;
- Chronos-2 zero-shot (variant `c2_lr_long_aef`, last week's curve as known future input), run a second time with
  every reading after `now` removed and one call per forecast moment; the second run is used.

Value = (1 - `track_weight`) x accepted + `track_weight` x timing-shift; then, for `chronos_cameras` only,
(1 - `chronos_weight`) x that value + `chronos_weight` x Chronos-2. Runtime: about 5 minutes.

In [ ]:
# Active FORECAST settings (configs/version7.json -> "forecast")
FORECAST_CONFIG = dict(CONFIG["forecast"])
assert 0 <= FORECAST_CONFIG["track_weight"] <= 1 and 0 <= FORECAST_CONFIG["chronos_weight"] <= 1
assert set(FORECAST_CONFIG["chronos_cameras"]) <= set("ABCDEFGH")
FORECAST_CONFIG

In [ ]:
# Load the frozen T05 covariances and Chronos-2, and compute the timing-shift and Chronos-2 values (nothing is fitted)
variant = FORECAST_CONFIG["chronos_variant"]
run_step("04a_timing_shift_forecast", "noran/tracks/forecast/make_test_predictions.py")
run_step("04b_chronos2", "noran/tracks/lib_forecast/run_lib.py", variant, "test")
run_step("04c_chronos2_isolated", "noran/tracks/lib_forecast/verify/rerun_iso.py", "test", variant)

In [ ]:
# Blend into the FORECAST values for all 2,660 forecast targets
w, cw = FORECAST_CONFIG["track_weight"], FORECAST_CONFIG["chronos_weight"]
run_step("04d_blend_track", "noran/blend_forecast.py", "--write", w, "--test-only")
run_step("04e_blend_chronos", "noran/tracks/lib_forecast/verify/make_others_blend.py", cw, "--test-only",
         "--base-weight", w, "--variant", variant, "--cameras", FORECAST_CONFIG["chronos_cameras"])
FORECAST_VALUES = f"outputs/noran/tracks/lib_forecast_verify/blend_others_w{cw:g}_test_values.csv"
mission_summary(FORECAST_VALUES, "forecast")

**Evaluate after editing** (local only). The forecast scores need the team's fixed-set files in
`outputs/local_score/` (`scripts/run_local_score.ps1`); this step was not rerun for this package.
```powershell
$env:PYTHONPATH = (Get-Location).Path; $env:HF_HUB_OFFLINE = "1"; $env:PYTHONUTF8 = "1"
$py = ".\.venv_lab\Scripts\python.exe"
& $py -X utf8 noran\tracks\forecast\eval_fixed.py --set fixed; & $py -X utf8 noran\tracks\forecast\eval_fixed.py --set mirror
& $py -X utf8 noran\tracks\forecast\eval_rolling.py
& $py -X utf8 noran\blend_forecast.py --write 0.5          # prints the score of each track weight
foreach ($s in "fixed", "mirror") { & $py -X utf8 noran\tracks\lib_forecast\run_lib.py c2_lr_long_aef $s
                                     & $py -X utf8 noran\tracks\lib_forecast\verify\rerun_iso.py $s }
& $py -X utf8 noran\tracks\lib_forecast\verify\make_others_blend.py 0.25
& $py -X utf8 noran\candidate.py score noran\configs\v7_tabicl_chronos.json
```

# 5. HIDDEN

Estimates a camera hidden for the whole recording (B and G on Friday 18 Sep, D on Saturday 19 Sep) at the same
minute. `noran/tracks/hidden_v4/model.py` takes version 3 (`noran/tracks/hidden_v3`: five estimates blended in log
scale, a ridge regression plus last week's residual at the prayer-aligned minute, a boosted model, the team's
tree model, the teammate's CatBoost for B and D, and the camera's own curve from last week scaled to today; camera
B is averaged with the baseline where D stands far above the other cameras) and joins TabICL v2 in log scale with
the share `tabicl_share` per camera. The models are fitted on the training recordings and the released test rows
when the step runs. Runtime: about 4 minutes.

In [ ]:
# Active HIDDEN settings (configs/version7.json -> "hidden")
HIDDEN_CONFIG = dict(CONFIG["hidden"])
assert all(0 <= s <= 1 for s in HIDDEN_CONFIG["tabicl_share"].values())
HIDDEN_CONFIG

In [ ]:
# Fit the mission module on the training data and predict every HIDDEN target (one process)
run_step("05_hidden", "noran/track_predict.py", "hidden", HIDDEN_CONFIG["track_module"], "--label", HIDDEN_CONFIG["label"],
         env={"NORAN_HIDDEN_V4_SHARE": json.dumps(HIDDEN_CONFIG["tabicl_share"])})

In [ ]:
# HIDDEN predictions against the accepted baseline
mission_summary(HIDDEN_CONFIG["label"], "hidden")

**Evaluate after editing** (local only): fixed set (needs `outputs/local_score/`), mirror set, and the real test
days with a released camera hidden.
```powershell
$env:PYTHONPATH = (Get-Location).Path; $env:HF_HUB_OFFLINE = "1"; $env:PYTHONUTF8 = "1"
foreach ($s in "fixed", "mirror", "testday") {
  .\.venv_lab\Scripts\python.exe -X utf8 noran\track_eval.py hidden noran\tracks\hidden_v4\model.py --set $s --label hidden_v4 }
```

# 6. HIDDEN-FUTURE

Predicts a hidden camera 1 to 15 minutes after `now`; nothing after `now` is read. `noran/tracks/hf_final/model.py`
(on top of `noran/tracks/hidden_future/`) takes the camera's own curve from the same weekday last week, aligns it to
today's prayer times and scales it by today's level of the visible cameras over the last 10 minutes. In the late
afternoon the baseline value is kept. Its settings are constants at the top of the two model files.
Runtime: under 1 minute.

In [ ]:
# Active HIDDEN-FUTURE settings (configs/version7.json -> "hidden-future")
HIDDEN_FUTURE_CONFIG = dict(CONFIG["hidden-future"])
HIDDEN_FUTURE_CONFIG

In [ ]:
# Fit and predict every HIDDEN-FUTURE target (one process)
run_step("06_hidden_future", "noran/track_predict.py", "hidden-future", HIDDEN_FUTURE_CONFIG["track_module"],
         "--label", HIDDEN_FUTURE_CONFIG["label"])

In [ ]:
mission_summary(HIDDEN_FUTURE_CONFIG["label"], "hidden-future")

**Evaluate after editing** (local only):
```powershell
$env:PYTHONPATH = (Get-Location).Path; $env:HF_HUB_OFFLINE = "1"; $env:PYTHONUTF8 = "1"
foreach ($s in "fixed", "mirror", "testday") {
  .\.venv_lab\Scripts\python.exe -X utf8 noran\track_eval.py hidden-future noran\tracks\hf_final\model.py --set $s --label hf_final }
```

# 7. MISSING

Fills a gap inside a recording of a camera that is normally visible. `noran/tracks/missing/model.py` starts from
the straight line across the gap and corrects it with the partner camera's readings inside the gap and the
camera's own level smoothed from both sides; its stacking weights are fitted on gaps cut in the same pattern as the
real ones when the step runs. Runtime: under 1 minute.

In [ ]:
# Active MISSING settings (configs/version7.json -> "missing")
MISSING_CONFIG = dict(CONFIG["missing"])
MISSING_CONFIG

In [ ]:
# Fit and predict every MISSING target (one process)
run_step("07_missing", "noran/track_predict.py", "missing", MISSING_CONFIG["track_module"], "--label", MISSING_CONFIG["label"])

In [ ]:
mission_summary(MISSING_CONFIG["label"], "missing")

**Evaluate after editing** (local only):
```powershell
$env:PYTHONPATH = (Get-Location).Path; $env:HF_HUB_OFFLINE = "1"; $env:PYTHONUTF8 = "1"
foreach ($s in "fixed", "mirror") {
  .\.venv_lab\Scripts\python.exe -X utf8 noran\track_eval.py missing noran\tracks\missing\model.py --set $s --label missing }
```

# 8. Assemble, verify and save the submission

1. `noran/candidate.py build` writes the base file from `noran/configs/v6_hidden_v3.json` through the team's task
   functions (`src/baseline_tasks.py`, `src/baseline_workflow.py`).
2. `noran/assemble.py` puts the values of sections 4 to 7 into it; `noran/check_candidate.py` compares every row
   with the mission file it came from.
3. The file is saved as `outputs/version7/submission_version7.csv` with `task_config.json` and
   `verification.json`, after these checks: 4,457 rows, all sample IDs in the sample order, unique IDs, finite and
   non-negative values, and the SHA-256 in `models/version7/manifest.json`. If
   `outputs/version7/reference/submission_v7_board_0.724.csv` exists, every value is also compared by ID.

With the settings of `configs/version7.json` the file must match the fingerprint; the notebook stops otherwise.
After an intended change, set `VERIFY_ACCEPTED = False` to save a candidate whose fingerprint differs.
Runtime: about 2.5 minutes.

In [ ]:
VERIFY_ACCEPTED = True
CONFIG_PATH = save_settings()
assembly = CONFIG["assembly"]
run_step("08a_build_base", "noran/candidate.py", "build", assembly["base_config"])
candidate_dir = ROOT / "outputs/noran/candidates" / assembly["candidate_name"]
if candidate_dir.exists():
    shutil.rmtree(candidate_dir)          # assemble.py never overwrites; this folder holds only this notebook's output
run_step("08b_assemble", "noran/assemble.py", assembly["candidate_name"], "--from", assembly["base_name"],
         "--hidden", HIDDEN_CONFIG["label"], "--hidden-future", HIDDEN_FUTURE_CONFIG["label"],
         "--missing", MISSING_CONFIG["label"], "--forecast", FORECAST_VALUES)
run_step("08c_check_rows", "noran/check_candidate.py", assembly["candidate_name"], "--hidden", HIDDEN_CONFIG["label"],
         "--hidden-future", HIDDEN_FUTURE_CONFIG["label"], "--missing", MISSING_CONFIG["label"],
         "--forecast", FORECAST_VALUES)

final = OUT / "submission_version7.csv"
shutil.copyfile(candidate_dir / f"submission_{assembly['candidate_name']}.csv", final)
submission = pd.read_csv(final)
expected = MANIFEST["accepted_submission_sha256"]
digest = sha256(final)
checks = {"rows_4457": len(submission) == 4457,
          "ids_in_sample_order": submission.id.tolist() == sample.id.tolist(),
          "ids_unique": bool(submission.id.is_unique),
          "values_finite": bool(np.isfinite(submission.people).all()),
          "values_non_negative": bool((submission.people >= 0).all())}
verification = dict(output=str(final.relative_to(ROOT)), output_sha256=digest, expected_sha256=expected,
                    sha256_matches_accepted=digest == expected, checks=checks,
                    settings=str(CONFIG_PATH.relative_to(ROOT)), step_seconds=TIMES,
                    total_step_seconds=round(sum(TIMES.values()), 1))
reference = OUT / "reference/submission_v7_board_0.724.csv"
if reference.exists():
    ref = pd.read_csv(reference).set_index("id").people.reindex(submission.id).to_numpy()
    diff = np.abs(submission.people.to_numpy() - ref)
    verification.update(reference=str(reference.relative_to(ROOT)), unequal_values=int((diff != 0).sum()),
                        max_absolute_difference=float(np.nanmax(diff)))
(OUT / "verification.json").write_text(json.dumps(verification, indent=2), encoding="utf-8")
print(json.dumps(verification, indent=2))
assert all(checks.values()), "Format checks failed; see outputs/version7/verification.json"
if VERIFY_ACCEPTED:
    assert digest == expected, "The file differs from the version 7 fingerprint; see outputs/version7/verification.json"
print("saved:", final)

### Contributing a change

1. Create a branch, e.g. `git switch -c task/hidden-update`, and change one mission: its settings in
   `configs/version7.json`, or its module under `noran/tracks/` (copy the folder, edit the copy and point
   `track_module` and `label` at it).
2. Run that mission's "evaluate after editing" commands and compare with the version 7 numbers in
   `docs/version7.md`.
3. Set `VERIFY_ACCEPTED = False` in section 8, then Restart Kernel and Run All. The new file is in
   `outputs/version7/`, its settings in `task_config.json`.
4. Review `git diff` and commit the source and settings; data and outputs stay out of Git. Never change the
   fingerprint in the manifest to hide a difference: update it only for a file that is accepted as the new version.